# Assignment 7.1: Music Genre and Composer Classification Using Deep Learning

| <div align="center"> Field </div>| <div align="center"> Details </div> |
|-------|---------|
| **Team Members** | Pros Loung, Nicole Rowley, Reema Eid |
| **Course** | AAI-511 Neural Networks and Deep Learning |
| **FINAL PROJECT** | Music Genre and Composer Classification Using Deep Learning |
| **GitHub Repository** | https://github.com/nicole-rowley-msaai/MS_AAI_511_Neural_Networks_DL |

## Team Contribution Map

| <div align="center"> Team member </div> | <div align="center"> Workstream </div> | <div align="center"> Sections Owned </div> |
|---|---|---|
| **Pros** | | |
| **Nicole** | |  |
| **Reema** |  | |
| **Team** | Final validation, Final Report, and GitHub submission | Notebook 2 final report sections and presentation |

# Introduction

Music is a form of art that is ubiquitous and has a rich history. Different composers have created music with their unique styles and compositions. However, identifying the composer of a particular piece of music can be a challenging task, especially for novice musicians or listeners. The proposed project aims to use deep learning techniques to identify the composer of a given piece of music accurately.

# Objective

The primary objective of this project is to develop a deep learning model that can predict the composer of a given musical score accurately. The project aims to accomplish this objective by using two deep learning techniques: Long Short-Term Memory (LSTM) and Convolutional Neural Network (CNN).

# Methodology

The proposed project will be implemented using the following steps:

1. Data Collection: Use the provided dataset of musical scores and associated composer labels.
2. Data Pre-processing: Convert scores into a model-ready representation (for example MIDI/event sequences) and apply quality checks and augmentation where appropriate.
3. Feature Extraction: Extract relevant musical features such as note patterns, chord progressions, rhythm, and tempo.
4. Model Building: Develop deep learning classifiers using both LSTM and CNN architectures.
5. Model Training: Train the models on preprocessed and feature-engineered data using consistent train/validation/test splits.
6. Model Evaluation: Evaluate performance using accuracy, precision, recall, and F1-score.
7. Model Optimization: Tune hyperparameters and regularization settings to improve generalization.

# Final Project Setup and Pipeline

1. Project Setup
- Define project scope, assumptions, and success criteria.
- Create reproducible environment and dependency list.
- Set random seeds and experiment tracking conventions.

2. Load Data and Exploration
- Load raw music-score data and labels.
- Check class balance, missing labels, and duplicate samples.
- Establish train/validation/test split strategy with stratification.

3. Representation and Preprocessing
- Convert source files to a consistent symbolic representation.
- Normalize sequence length strategy (padding/truncation/windows).
- Build preprocessing pipeline that can be reused for inference.

4. Feature Engineering
- Build two branches of inputs: sequence-based inputs for LSTM and image/grid-like representations for CNN (if applicable).
- Generate optional handcrafted features (tempo, pitch range, interval stats) for comparison baselines.

5. Deep Model Development
- LSTM track: embedding/sequence layers, recurrent blocks, dropout, dense classifier.
- CNN track: convolution blocks, pooling, normalization, dense classifier.
- Keep model definitions modular to support ablation studies.

- **Training and Validation**
    - Use early stopping and model checkpointing.
    - Run controlled experiments across learning rate, batch size, depth, and dropout.
    - Track every run with config, metrics, and notes.

- **Evaluation and Error Analysis**
    - Evaluate best models on hold-out test set.
    - Report accuracy, precision, recall, F1-score, and confusion matrix.
    - Analyze frequent misclassifications and class-specific weaknesses.

6. Model Selection and Optimization
- Compare LSTM vs CNN against objective metrics and training stability.
- Select final model using both metric quality and robustness.
- Apply final hyperparameter tuning pass.

7. Packaging and Reproducibility
- Save preprocessing artifacts, label encoder, and model weights.
- Create an inference function for unseen music input.
- Document end-to-end run steps for reproducibility.

8. Final Deliverables
- Final notebook with narrative and visual results.
- Short report: problem, method, results, limitations, future work.

# Pipeline Connection Diagram

```mermaid
flowchart TD
    A[1. Project Setup] --> B[2. Load Data and Exploration]
    B --> C[3. Representation and Preprocessing]
    C --> D[4. Feature Engineering]
    D --> F[5. Deep Model Development<br/>LSTM and CNN Tracks<br/>Training, Validation, and Error Analysis]
    F --> I[6. Model Selection and Optimization]
    I --> J[7. Packaging and Reproducibility]
    J --> K[8. Final Deliverables]

    F -. Compare architectures .-> I
    F -. Feedback loop for improvements .-> C
    I -. Retune if needed .-> F
```

## 1.0 Project Setup
### Owner: Pros


In [15]:
# Install libraries
# Use the following command to install the pretty_midi library if not already installed
#!pip install pretty_midi
#!pip install kagglehub
#!pip install pygame

In [16]:
# Load Library 
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn

## 2. Load Data and Exploration
### Onwer: Pros

- Load raw music-score data and labels.
- Check class balance, missing labels, and duplicate samples.
- Establish train/validation/test split strategy with stratification.

In [17]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("blanderbuss/midi-classic-music")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Loung\.cache\kagglehub\datasets\blanderbuss\midi-classic-music\versions\1


In [18]:
# Load MIDI files directly from midiclassics.zip and create reproducible splits
from pathlib import Path
from collections import Counter
import zipfile
import pretty_midi
from sklearn.model_selection import train_test_split

data_root = Path(r"C:\Users\Loung\USD\AAI-551 Neural Networks\MS_AAI_511_Neural_Networks_DL")
zip_path = data_root / "midiclassics.zip"
extract_root = data_root / "midiclassics_extracted"

if not zip_path.exists():
    raise FileNotFoundError(f"Could not find dataset archive: {zip_path}")

if not extract_root.exists():
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_root)

# Only load these composers
target_composers = ["Bach", "Beethoven", "Chopin", "Mozart"]
target_composers_lower = {name.lower() for name in target_composers}

composer_files = []
for composer_dir in extract_root.iterdir():
    if not composer_dir.is_dir() or composer_dir.name.lower() not in target_composers_lower:
        continue

    for midi_file in composer_dir.rglob("*.mid"):
        composer_files.append(
            {
                "path": str(midi_file),
                "composer": composer_dir.name,
            }
        )

if not composer_files:
    raise ValueError("No target composer MIDI files were found in the extracted archive.")

labels = [item["composer"].lower() for item in composer_files]
train_records, temp_records = train_test_split(
    composer_files,
    test_size=0.30,
    random_state=42,
    stratify=labels,
)
temp_labels = [item["composer"].lower() for item in temp_records]
dev_records, test_records = train_test_split(
    temp_records,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels,
)

split_records = {
    "train": train_records,
    "dev": dev_records,
    "test": test_records,
}
midi_data = {"dev": [], "train": [], "test": []}
load_errors = []

for split, records in split_records.items():
    for record in records:
        midi_file = Path(record["path"])
        try:
            pm = pretty_midi.PrettyMIDI(str(midi_file))
            midi_data[split].append(
                {
                    "path": str(midi_file),
                    "composer": record["composer"],
                    "midi": pm,
                }
            )
        except Exception as e:
            load_errors.append((str(midi_file), str(e)))

print(f"Using dataset archive: {zip_path.name}")
print(f"Extracted files directory: {extract_root}")

# Convenience lists if you want to keep your previous variable names
# midi_dev = [item["midi"] for item in midi_data["dev"]]
# midi_train = [item["midi"] for item in midi_data["train"]]
# midi_test = [item["midi"] for item in midi_data["test"]]

# Quick verification output
for split in ["train", "dev", "test"]:
    composer_counts = Counter(item["composer"].lower() for item in midi_data[split])
    print(f"{split.upper()} files loaded: {len(midi_data[split])}")
    for composer_name in target_composers:
        print(f"  - {composer_name}: {composer_counts.get(composer_name.lower(), 0)}")

if load_errors:
    print(f"\nFiles with load errors: {len(load_errors)}")
    for file_path, err in load_errors[:5]:
        print(f"  - {file_path}: {err}")
        

c:\Users\Loung\USD\AAI-530-IoT\AAI-530-IoT-Assignment-4.1\.venv\Lib\site-packages\pretty_midi\pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


Using dataset archive: midiclassics.zip
Extracted files directory: C:\Users\Loung\USD\AAI-551 Neural Networks\MS_AAI_511_Neural_Networks_DL\midiclassics_extracted
TRAIN files loaded: 1140
  - Bach: 717
  - Beethoven: 149
  - Chopin: 95
  - Mozart: 179
DEV files loaded: 243
  - Bach: 153
  - Beethoven: 31
  - Chopin: 21
  - Mozart: 38
TEST files loaded: 245
  - Bach: 154
  - Beethoven: 32
  - Chopin: 20
  - Mozart: 39

Files with load errors: 2
  - C:\Users\Loung\USD\AAI-551 Neural Networks\MS_AAI_511_Neural_Networks_DL\midiclassics_extracted\Mozart\Piano Sonatas\Nueva carpeta\K281 Piano Sonata n03 3mov.mid: Could not decode key with 2 flats and mode 2
  - C:\Users\Loung\USD\AAI-551 Neural Networks\MS_AAI_511_Neural_Networks_DL\midiclassics_extracted\Beethoven\Anhang 14-3.mid: Could not decode key with 3 flats and mode 255


In [19]:
# Convenience lists if you want to keep your previous variable names
midi_dev = [item["midi"] for item in midi_data["dev"]]
midi_train = [item["midi"] for item in midi_data["train"]]
midi_test = [item["midi"] for item in midi_data["test"]]

In [20]:
print(f"Number of MIDI files in dev set: {len(midi_dev)}")

print(f"Number of MIDI files in train set: {len(midi_train)}")

print(f"Number of MIDI files in test set: {len(midi_test)}")

# Display the first few MIDI files in each set
print("First few MIDI files in dev set:")
for midi_file in midi_dev[:5]:
    print(midi_file)

print("\nFirst few MIDI files in train set:")
for midi_file in midi_train[:5]:
    print(midi_file)

print("\nFirst few MIDI files in test set:")
for midi_file in midi_test[:5]:
    print(midi_file)

Number of MIDI files in dev set: 243
Number of MIDI files in train set: 1140
Number of MIDI files in test set: 245
First few MIDI files in dev set:

First few MIDI files in train set:

First few MIDI files in test set:


In [21]:
# Preview the first MIDI file in the dev set as synthesized audio
from IPython.display import Audio, display

sample_rate = 16000
audio_waveform = midi_dev[0].synthesize(fs=sample_rate)
display(Audio(audio_waveform, rate=sample_rate))
print("Displayed an audio preview for the first MIDI file in the dev set.")

Displayed an audio preview for the first MIDI file in the dev set.


## 3. Representation and Preprocessing
### Owner: Pros
- Convert source files to a consistent symbolic representation.
- Normalize sequence length strategy (padding/truncation/windows).
- Build preprocessing pipeline that can be reused for inference.

## 4. Feature Engineering
### Owner: Pros
- Build two branches of inputs: sequence-based inputs for LSTM and image/grid-like representations for CNN (if applicable).
- Generate optional handcrafted features (tempo, pitch range, interval stats) for comparison baselines.

## 5. Deep Models Development

### 5.1. LSTM Model Development
### Owner: Reema

In [22]:
# Enter your code here to define the LSTM model architecture, including embedding/sequence layers, recurrent blocks, dropout, and a dense classifier. Make sure to keep the model definitions modular to facilitate ablation studies.

### Training, Optimization, and Validation

In [23]:

# Use early stopping and model checkpointing.
# Run controlled experiments across learning rate, batch size, depth, and dropout.
# Track every run with config, metrics, and notes.

### Evaluation and Error Analysis

In [24]:
# Evaluate best models on hold-out test set.
# Report accuracy, precision, recall, F1-score, and confusion matrix.
# Analyze frequent misclassifications and class-specific weaknesses.

### 5.2. CNN Model Development
### Owner: Nicole

In [25]:
# Enter your code here to define the CNN model architecture, including convolutional blocks, pooling layers, normalization, and a dense classifier. Make sure to keep the model definitions modular to facilitate ablation studies.

###  Training, Optimization and Validation

In [26]:
# Use early stopping and model checkpointing.
# Run controlled experiments across learning rate, batch size, depth, and dropout.
# Track every run with config, metrics, and notes.

### Evaluation and Error Analysis

In [27]:
# Evaluate best models on hold-out test set.
# Report accuracy, precision, recall, F1-score, and confusion matrix.
# Analyze frequent misclassifications and class-specific weaknesses.

## 6. Model Selection and Optimization
### Owner: Reema and Nicole
- Compare LSTM vs CNN against objective metrics and training stability.
- Select final model using both metric quality and robustness.
- Apply final hyperparameter tuning pass.

## 7. Packaging and Reproducibility
### Owner: Reema or Nicole depending on which model is selected.

In [28]:
# Save preprocessing artifacts, label encoder, and model weights.
# Create an inference function for unseen music input.
# Document end-to-end run steps for reproducibility.

### Optional demo cell: predict composer for a sample input.

## 8. Final Deliverables
### Owner: Pros
- Final notebook with narrative and visual results.

### Owner: Team
- Final Report: problem, method, results, limitations, future work.
